# A→B вокруг стены: continuous conditions и две допустимые моды

Для **одного и того же** `[Axy, Bxy, wall_low_xy, wall_high_xy]` dataset содержит верхние
и нижние обходы. **Route ID не подаётся модели.** Это более трудная условная задача:
усреднение двух маршрутов может пройти прямо сквозь стену. Проверяем exact segment
collisions и сохранение обеих мод, а не только MSE к teacher.

Модель `PointDenoiser` предсказывает **30 внутренних точек** полного пути из 32 точек.
Начало A и конец B добавляются из condition на каждом Euler-узле. **Exact endpoints —
конструкция, не выученное достижение.** State dimension для MSE/OT — 60 координат.
Никаких obstacle penalties, проекции пути на свободную область или postprocessing нет.

Teacher: pure conditional FM, 20 000 updates, hidden128 × 3, continuous MLP-conditioner8.
`t=1` — шум, `t=0` — данные; `PointDenoiser` получает `t*1000`. Null и CFG здесь не
предлагаются: teacher обучается без condition dropout. Default загружает отдельно
опубликованный native teacher; `TRAIN_TEACHER=True` показывает обучение с нуля.

Default student-пилот — **200 updates, один seed, 32→4**, M0=MSE и E0=MSE+Sinkhorn.
Для более длинного exploratory-прогона задайте 1000 updates, warmup100; для 32→2 — новый
output directory и `STUDENT_STEPS=2`. Это отдельный wall-пилот, не замена зарегистрированных
unconditional результатов. Веса, wheel и outputs не встроены в notebook.

In [ ]:
import os
import subprocess
import sys
from importlib.metadata import version

from packaging.version import Version

GIT_BRANCH = "research/wasserstein-tube-replication"
MIN_VERSION = "0.4.4"
GIT_SOURCE = ("diffusion-lab[replication,interactive] @ "
              "git+https://github.com/Vovanm88/ddpm_vibe_research.git@" + GIT_BRANCH)
if os.environ.get("DIFFUSION_LAB_SKIP_INSTALL") != "1":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", GIT_SOURCE])
if Version(version("diffusion-lab")) < Version(MIN_VERSION):
    raise RuntimeError("Install diffusion-lab >=0.4.4 from this branch and restart the kernel")


In [ ]:
import copy
import hashlib
import json
import math
import tempfile
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import torch
from IPython.display import Image, clear_output, display
from tqdm.auto import tqdm

from diffusion_lab import PointDenoiser
from diffusion_lab.environments.multimodal_navigation import (
    evaluate_navigation,
    make_multimodal_navigation,
)
from diffusion_lab.navigation_visualization import (
    plot_navigation_denoising,
    plot_navigation_paths,
    save_navigation_denoising_gif,
    save_navigation_particle_gif,
)
from diffusion_lab.visualization import save_figure
from recheck.calibration import calibrate_interval_features
from recheck.fm import learning_rate
from recheck.losses import production_sinkhorn, sliced_wasserstein_loss
from recheck.runtime import (
    RNGStreams,
    atomic_json,
    choose_device,
    model_hash,
    save_checkpoint,
    tensor_hash,
)


In [ ]:
DEVICE = "cpu"
THREADS = 4
DATA_SEED = 61
VALIDATION_SEED = 10061
TRAIN_SCENES = 128
VALIDATION_SCENES = 32
PATHS_PER_SCENE = 128
PATH_LENGTH = 32
JITTER = .025
CLEARANCE = .06
COLLISION_MARGIN = .035
HIDDEN_DIM = 128
NUM_LAYERS = 3
TRAIN_TIMESTEPS = 1000

TRAIN_TEACHER = False
TEACHER_SEED = 61
TEACHER_UPDATES = 20000
TEACHER_BATCH_SIZE = 256
TEACHER_LR = 1e-3
TEACHER_WARMUP = 1000
TEACHER_EMA = .999
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 1.

SEED = 161
UPDATES = 200                       # exploratory; set 1000 and WARMUP=100 for longer pilot
BATCH_SIZE = 128                    # whole OT support has one continuous scene condition
LEARNING_RATE = 1e-4
WARMUP = 20
EMA_DECAY = .99
TEACHER_STEPS = 32
STUDENT_STEPS = 4                   # 2: new calibration + students in a new OUTPUT_DIR
DISTRIBUTION_FAMILY = "sinkhorn"    # or "swd"
REPRESENTATION = "endpoint"        # or displacement/joint, always interior D=60
LAMBDA_W = 1.
GAMMA = 1.
EPSILON_ALPHA = .05
CALIBRATION_SAMPLES = 128
CALIBRATION_SEED = 610014
SWD_PROJECTIONS = 128
CHECKPOINT_EVERY = 50
RESUME = False

TEACHER_NFE = (1, 2, 4, 8, 16, 32, 64)
EVAL_SCENES = 4                     # first fixed held-out scenes; full pilot uses32
SAMPLES_PER_SCENE = 64              # full pilot uses128
EVAL_SEED = 613001
SAVE_GIFS = True
SHOW_WIDGETS = True
PROGRESS = True
PREVIEW_PATHS = 4
OUTPUT_DIR = Path("replication/artifacts/wall_navigation_notebook14")
PUBLISHED_TEACHER_PATH = Path("replication/demo/wall_navigation_conditional_v1/teacher/ema_model")


In [ ]:
def expected_teacher_profile():
    return {"data": {"n_scenes": TRAIN_SCENES, "samples_per_scene": PATHS_PER_SCENE,
            "path_length": PATH_LENGTH, "seed": DATA_SEED, "split": "train",
            "jitter": JITTER, "clearance": CLEARANCE, "margin": COLLISION_MARGIN},
        "model": {"sample_shape": [PATH_LENGTH - 2, 2], "hidden_dim": HIDDEN_DIM,
            "num_layers": NUM_LAYERS, "backbone": "mlp", "conditioner": "mlp",
            "condition_dim": 8, "num_classes": 0},
        "teacher": {"seed": TEACHER_SEED, "updates": TEACHER_UPDATES,
            "batch_size": TEACHER_BATCH_SIZE, "lr": TEACHER_LR, "weight_decay": WEIGHT_DECAY,
            "warmup": TEACHER_WARMUP, "ema_decay": TEACHER_EMA,
            "grad_clip": GRAD_CLIP, "condition_dropout": 0.}}


def load_navigation_parent(candidates, expected, device):
    rejected = []
    for candidate in candidates:
        candidate = Path(candidate)
        try:
            manifest = json.loads((candidate.parent / "manifest.json").read_text())
            native = json.loads((candidate / "config.json").read_text())
            if manifest.get("status") != "completed" or manifest.get("updates_completed", 0) < 20000:
                raise ValueError("wall teacher has not completed20000 updates")
            for section in ("data", "model", "teacher"):
                if manifest["config"][section] != expected[section]:
                    raise ValueError(f"wrong wall teacher {section} profile")
            if not manifest.get("endpoint_clamped_contract"):
                raise ValueError("missing explicit endpoint reconstruction contract")
            if native.get("_class_name") != "PointDenoiser":
                raise ValueError("wrong native model class")
            for key, value in expected["model"].items():
                if native.get(key) != value:
                    raise ValueError(f"wrong continuous model setting {key}")
            model = PointDenoiser.from_pretrained(candidate,local_files_only=True,
                low_cpu_mem_usage=False).to(device).eval().requires_grad_(False)
            if model_hash(model) != manifest["teacher_sha256"]:
                raise ValueError("wall parent hash mismatch")
            return model, candidate, manifest
        except (OSError,ValueError,KeyError,TypeError) as error:
            rejected.append(f"{candidate}: {error}")
    raise FileNotFoundError("Train/publish the matching continuous wall teacher, or set TRAIN_TEACHER=True.\n"
                            + "\n".join(rejected))


In [ ]:
repo_dir = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "src/diffusion_lab").is_dir()), None)
if repo_dir is None:
    repo_dir = Path(tempfile.mkdtemp(prefix="wall-navigation-source-")) / "repository"
    subprocess.check_call(["git","clone","--depth","1","--branch",GIT_BRANCH,
                           "https://github.com/Vovanm88/ddpm_vibe_research.git",str(repo_dir)])
torch.set_num_threads(THREADS)
device = choose_device(DEVICE)
OUTPUT_DIR = Path(OUTPUT_DIR).expanduser().resolve()
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
profile = expected_teacher_profile()
train = make_multimodal_navigation(**profile["data"])
validation = make_multimodal_navigation(n_scenes=VALIDATION_SCENES,samples_per_scene=PATHS_PER_SCENE,
    path_length=PATH_LENGTH,seed=VALIDATION_SEED,split="validation",
    jitter=JITTER,clearance=CLEARANCE,margin=COLLISION_MARGIN)
train_scene_indices = {scene:torch.where(train.scene_indices==scene)[0] for scene in range(TRAIN_SCENES)}
assert train.conditions.shape[1]==8 and train.conditions.is_floating_point()
assert TEACHER_STEPS % STUDENT_STEPS==0 and STUDENT_STEPS>0
MICROSTEPS,H = TEACHER_STEPS//STUDENT_STEPS,1/STUDENT_STEPS
print("Each condition contains BOTH top/bottom reference routes. Route IDs never enter the model.")
print("State:",train.samples[:,1:-1].shape,"continuous conditions:",train.conditions.shape)


## Видимый pure-CFM teacher на внутренних точках

У каждого примера continuous condition8; route labels в training не читаются.
Подстановка A/B происходит позже при реконструкции полного пути, не в loss и не через
obstacle projection. Default пропускает этот долгий цикл и проверяет native20k parent.

In [ ]:
if TRAIN_TEACHER:
    teacher_dir = OUTPUT_DIR / "teacher"
    if teacher_dir.exists() and any(teacher_dir.iterdir()):
        raise FileExistsError("Choose a new OUTPUT_DIR for a new wall teacher")
    teacher_dir.mkdir(parents=True,exist_ok=True)
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(TEACHER_SEED)
        teacher_model = PointDenoiser(**profile["model"]).to(device)
    teacher_ema = copy.deepcopy(teacher_model).eval().requires_grad_(False)
    teacher_optimizer = torch.optim.AdamW(teacher_model.parameters(),lr=TEACHER_LR,
        betas=(.9,.999),weight_decay=WEIGHT_DECAY)
    teacher_streams = RNGStreams(TEACHER_SEED)
    teacher_history=[]
    for update in tqdm(range(TEACHER_UPDATES),desc="Wall conditional FM",disable=not PROGRESS):
        indices=torch.randint(len(train),(TEACHER_BATCH_SIZE,),generator=teacher_streams["data"])
        clean=train.samples[indices,1:-1].to(device)
        conditions=train.conditions[indices].to(device)
        noise=torch.randn(clean.shape,generator=teacher_streams["noise"]).to(device)
        t=torch.rand(len(clean),generator=teacher_streams["interval"]).to(device)
        anchor=(1-t[:,None,None])*clean+t[:,None,None]*noise
        prediction=teacher_model(anchor,t*TRAIN_TIMESTEPS,conditions=conditions).sample
        loss=(prediction-(noise-clean)).square().mean()
        if not torch.isfinite(loss):
            raise FloatingPointError("Nonfinite wall teacher loss")
        lr=learning_rate(update,TEACHER_UPDATES,TEACHER_WARMUP,TEACHER_LR)
        for group in teacher_optimizer.param_groups:
            group["lr"]=lr
        teacher_optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(teacher_model.parameters(),GRAD_CLIP,error_if_nonfinite=True)
        teacher_optimizer.step()
        with torch.no_grad():
            for average,parameter in zip(teacher_ema.parameters(),teacher_model.parameters()):
                average.lerp_(parameter.detach(),1-TEACHER_EMA)
            for average,buffer in zip(teacher_ema.buffers(),teacher_model.buffers()):
                average.copy_(buffer)
        teacher_history.append({"update":update+1,"loss":float(loss.detach()),"lr":lr})
    teacher=teacher_ema.eval().requires_grad_(False)
    teacher_path=teacher_dir/"ema_model"
    teacher.save_pretrained(teacher_path,safe_serialization=True)
    teacher_manifest={"status":"completed","config":profile,"updates_completed":TEACHER_UPDATES,
        "teacher_sha256":model_hash(teacher),"data_sha256":tensor_hash(train.samples),
        "conditions_sha256":tensor_hash(train.conditions),"endpoint_clamped_contract":True}
    atomic_json(teacher_dir/"manifest.json",teacher_manifest)
    atomic_json(teacher_dir/"history.json",teacher_history)
else:
    teacher,teacher_path,teacher_manifest=load_navigation_parent([
        repo_dir/"replication/artifacts/wall_navigation_conditional_v1/teacher/ema_model",
        repo_dir/PUBLISHED_TEACHER_PATH],profile,device)
    if teacher_manifest["data_sha256"]!=tensor_hash(train.samples):
        raise ValueError("Wall train-data hash differs")
    if teacher_manifest["conditions_sha256"]!=tensor_hash(train.conditions):
        raise ValueError("Wall continuous-condition hash differs")
teacher_hash=model_hash(teacher)
print("Frozen wall parent:",teacher_path,teacher_hash)


## Sampling: continuous condition передаётся в каждый Euler-вызов

Шум имеет форму `[B,30,2]`. Полная spatial path получается добавлением A/B.
FM denoising time и движение частицы по финальному пути — разные процессы; ниже будут
**два отдельных GIF**. Непрерывные отрезки проверяются на пересечение стены, включая
пересечение между сохранёнными waypoints.

In [ ]:
def reconstruct_endpoints(interior,conditions):
    conditions=torch.as_tensor(conditions,device=interior.device,dtype=interior.dtype)
    return torch.cat((conditions[:,:2,None].transpose(1,2),interior,
                      conditions[:,2:4,None].transpose(1,2)),dim=1)


@torch.no_grad()
def teacher_microsteps(model,anchor,t,width,conditions,microsteps):
    state=anchor.detach()
    h=width/microsteps
    for k in range(microsteps):
        velocity=model(state,(float(t)-k*h)*TRAIN_TIMESTEPS,conditions=conditions).sample
        state=state-h*velocity
    return state.detach()


def transition_features(anchor,endpoint,representation="endpoint",gamma=1.):
    dimension=endpoint[0].numel()     # 30×2=60, excludes constructed endpoints
    a,y=anchor.flatten(1),endpoint.flatten(1)
    if representation=="endpoint":
        return y/math.sqrt(dimension)
    if representation=="displacement":
        return (y-a)/math.sqrt(dimension)
    if representation=="joint":
        return torch.cat((math.sqrt(gamma)*a,y-a),dim=1)/math.sqrt(dimension)
    raise ValueError("unknown representation")


@torch.no_grad()
def sample_navigation(model,initial_noise,conditions,steps=32):
    if not isinstance(steps,int) or steps<1:
        raise ValueError("steps must be a positive integer")
    if conditions is None:
        raise ValueError("This teacher has no trained null branch: supply continuous condition8")
    conditions=torch.as_tensor(conditions,device=model.device,dtype=model.dtype)
    if conditions.ndim==1:
        conditions=conditions[None].expand(len(initial_noise),-1)
    if conditions.shape!=(len(initial_noise),8) or not torch.isfinite(conditions).all():
        raise ValueError("conditions must be finite [B,8] or [8]")
    if initial_noise.shape!=(len(conditions),PATH_LENGTH-2,2):
        raise ValueError("initial_noise must be [B,30,2]")
    was_training=model.training
    model.eval()
    try:
        state=initial_noise.detach().to(device=model.device,dtype=model.dtype).clone()
        history=[reconstruct_endpoints(state,conditions).cpu().clone()]
        for step in range(steps):
            t=1-step/steps
            velocity=model(state,t*TRAIN_TIMESTEPS,conditions=conditions).sample
            state=state-velocity/steps
            if not torch.isfinite(state).all():
                raise FloatingPointError("Nonfinite navigation trajectory")
            history.append(reconstruct_endpoints(state,conditions).cpu().clone())
    finally:
        model.train(was_training)
    return {"samples":history[-1],"history":torch.stack(history),"times":torch.linspace(1,0,steps+1),
            "initial_noise_sha256":tensor_hash(initial_noise),"conditions_sha256":tensor_hash(conditions),
            "nfe":steps,"endpoint_clamped_contract":True}


## Calibration: только train scenes, отдельно scene × interval

Support состоит из разных верхних/нижних путей **одной и той же geometry condition**.
Scene index используется только для выбора данных; модель получает исключительно 8 floats.
Teacher features — вся внутренняя path как один вектор D60, а не 30 независимых точек.

In [ ]:
calibration_streams=RNGStreams(CALIBRATION_SEED)
calibration={}
for scene_id in tqdm(range(TRAIN_SCENES),desc="Train-only scene calibration",disable=not PROGRESS):
    pool=train_scene_indices[scene_id]
    records=[]
    for j in range(STUDENT_STEPS):
        indices=pool[torch.randint(len(pool),(CALIBRATION_SAMPLES,),generator=calibration_streams["data"])]
        clean=train.samples[indices,1:-1].to(device)
        conditions=train.conditions[indices].to(device)
        assert torch.equal(conditions,conditions[:1].expand_as(conditions))
        noise=torch.randn(clean.shape,generator=calibration_streams["noise"]).to(device)
        t=1-j*H
        anchor=(1-t)*clean+t*noise
        endpoint=teacher_microsteps(teacher,anchor,t,H,conditions,MICROSTEPS)
        phi=transition_features(anchor,endpoint,REPRESENTATION,GAMMA)
        scale=calibrate_interval_features(phi.detach().cpu().double(),alpha=EPSILON_ALPHA)
        records.append({"interval":j,"t":t,"H":H,"scene_id":scene_id,**scale})
    calibration[str(scene_id)]=records
calibration_metadata={"teacher_sha256":teacher_hash,"train_sha256":tensor_hash(train.samples),
    "conditions_sha256":tensor_hash(train.conditions),"representation":REPRESENTATION,
    "gamma":GAMMA,"alpha":EPSILON_ALPHA,"support_size":CALIBRATION_SAMPLES,
    "seed":CALIBRATION_SEED,"scene_interval_table":calibration}
calibration_hash=hashlib.sha256(json.dumps(calibration_metadata,sort_keys=True).encode()).hexdigest()
atomic_json(OUTPUT_DIR/"calibration.json",calibration_metadata|{"sha256":calibration_hash})


## Видимый MSE / Sinkhorn student loop

Два режима начинают из одних teacher-весов, видят одинаковые scene/time/data/noise streams.
Вся OT-матрица относится к одной непрерывной condition. Sinkhorn — strict float64 solver
со student self-term gradients. Ни collision penalty, ни endpoint penalty сюда не добавляются.

In [ ]:
method_specs=[("M0",None),("E0",DISTRIBUTION_FAMILY)]
trained,training_logs,pairing_traces={},{},{}
for method_name,family in method_specs:
    student=copy.deepcopy(teacher).train().requires_grad_(True)
    ema=copy.deepcopy(teacher).eval().requires_grad_(False)
    assert model_hash(student)==model_hash(ema)==teacher_hash
    optimizer=torch.optim.AdamW(student.parameters(),lr=LEARNING_RATE,betas=(.9,.999),weight_decay=WEIGHT_DECAY)
    streams=RNGStreams(SEED)
    scene_generator=torch.Generator().manual_seed(SEED+14000)
    method_dir=OUTPUT_DIR/method_name
    method_dir.mkdir(parents=True,exist_ok=True)
    checkpoint_path=method_dir/"training.pt"
    contract={"scope":"one_seed_exploratory_multimodal_navigation","teacher_sha256":teacher_hash,
        "train_sha256":tensor_hash(train.samples),"conditions_sha256":tensor_hash(train.conditions),
        "seed":SEED,"updates":UPDATES,"batch_size":BATCH_SIZE,"teacher_steps":TEACHER_STEPS,
        "student_steps":STUDENT_STEPS,"family":family,"representation":REPRESENTATION,
        "lambda_w":LAMBDA_W,"gamma":GAMMA,"lr":LEARNING_RATE,"warmup":WARMUP,
        "weight_decay":WEIGHT_DECAY,"ema_decay":EMA_DECAY,"grad_clip":GRAD_CLIP,
        "swd_projections":SWD_PROJECTIONS,"calibration_sha256":calibration_hash,
        "endpoint_clamped_contract":True,"state_dimension":(PATH_LENGTH-2)*2,"route_id_input":False}
    completed,history,pairing_trace=0,[],[]
    if checkpoint_path.exists():
        if not RESUME:
            raise FileExistsError("Choose a new OUTPUT_DIR or RESUME=True")
        saved=torch.load(checkpoint_path,map_location="cpu",weights_only=True)
        if saved["contract"]!=contract:
            raise ValueError("Wall student resume contract differs")
        student.load_state_dict(saved["student"])
        ema.load_state_dict(saved["ema"])
        optimizer.load_state_dict(saved["optimizer"])
        for state in optimizer.state.values():
            for key,value in state.items():
                if isinstance(value,torch.Tensor):
                    state[key]=value.to(device)
        streams.load_state_dict(saved["streams"])
        scene_generator.set_state(saved["scene_rng"])
        completed,history,pairing_trace=saved["updates_completed"],saved["history"],saved["pairing_trace"]

    def commit(status):
        save_checkpoint(checkpoint_path,{"contract":contract,"student":student.state_dict(),
            "ema":ema.state_dict(),"optimizer":optimizer.state_dict(),"streams":streams.state_dict(),
            "scene_rng":scene_generator.get_state(),"updates_completed":completed,
            "history":history,"pairing_trace":pairing_trace,"status":status})

    commit("running")
    try:
        for update in tqdm(range(completed,UPDATES),desc=method_name,disable=not PROGRESS):
            scene_id=int(torch.randint(TRAIN_SCENES,(),generator=scene_generator))
            pool=train_scene_indices[scene_id]
            indices=pool[torch.randint(len(pool),(BATCH_SIZE,),generator=streams["data"])]
            clean=train.samples[indices,1:-1].to(device)
            conditions=train.conditions[indices].to(device)
            assert torch.equal(conditions,conditions[:1].expand_as(conditions))
            j=int(torch.randint(STUDENT_STEPS,(),generator=streams["interval"]))
            t=1-j*H
            noise=torch.randn(clean.shape,generator=streams["noise"]).to(device)
            anchor=(1-t)*clean+t*noise
            teacher_endpoint=teacher_microsteps(teacher,anchor,t,H,conditions,MICROSTEPS)
            student_endpoint=anchor-H*student(anchor,t*TRAIN_TIMESTEPS,conditions=conditions).sample
            pair=(student_endpoint-teacher_endpoint.detach()).square().mean()
            dist=pair.new_zeros(())
            if family is not None and LAMBDA_W!=0:
                phi_student=transition_features(anchor,student_endpoint,REPRESENTATION,GAMMA)
                phi_teacher=transition_features(anchor,teacher_endpoint,REPRESENTATION,GAMMA).detach()
                if family=="sinkhorn":
                    dist=production_sinkhorn(phi_student.float(),phi_teacher.float(),calibration[str(scene_id)][j]["epsilon"])
                elif family=="swd":
                    dist=sliced_wasserstein_loss(phi_student.float(),phi_teacher.float(),
                        num_directions=SWD_PROJECTIONS,generator=streams["swd"])
                else:
                    raise ValueError("unknown family")
            loss=pair+LAMBDA_W*dist
            if not torch.isfinite(loss):
                raise FloatingPointError("Nonfinite wall student loss")
            lr=learning_rate(update,UPDATES,WARMUP,LEARNING_RATE)
            for group in optimizer.param_groups:
                group["lr"]=lr
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            grad_norm=torch.nn.utils.clip_grad_norm_(student.parameters(),GRAD_CLIP,error_if_nonfinite=True)
            optimizer.step()
            with torch.no_grad():
                for average,parameter in zip(ema.parameters(),student.parameters()):
                    average.lerp_(parameter.detach(),1-EMA_DECAY)
                for average,buffer in zip(ema.buffers(),student.buffers()):
                    average.copy_(buffer)
            completed=update+1
            pairing_trace.append(hashlib.sha256((f"{scene_id}:{j}:"+tensor_hash(indices)+tensor_hash(noise)).encode()).hexdigest())
            history.append({"update":completed,"scene_id":scene_id,"interval":j,
                "pair_loss":float(pair.detach()),"dist_loss":float(dist.detach()),
                "total_loss":float(loss.detach()),"lr":lr,"grad_norm":float(grad_norm)})
            if completed==1 or completed%CHECKPOINT_EVERY==0:
                commit("running")
    except BaseException as error:
        saved=torch.load(checkpoint_path,map_location="cpu",weights_only=True)
        saved["status"]="interrupted" if isinstance(error,KeyboardInterrupt) else "failed"
        save_checkpoint(checkpoint_path,saved)
        atomic_json(method_dir/"manifest.json",contract|{"status":saved["status"],"updates_completed":saved["updates_completed"],"error":str(error)})
        raise
    commit("completed")
    ema.save_pretrained(method_dir/"ema_model",safe_serialization=True)
    atomic_json(method_dir/"manifest.json",contract|{"status":"completed","updates_completed":completed,
        "ema_sha256":model_hash(ema),"whole_support_single_condition":True})
    atomic_json(method_dir/"history.json",history)
    assert model_hash(teacher)==teacher_hash and all(p.grad is None for p in teacher.parameters())
    trained[method_name],training_logs[method_name],pairing_traces[method_name]=ema,history,pairing_trace
assert pairing_traces["M0"]==pairing_traces["E0"]
print("Paired scene/data/time/noise streams match; parent unchanged.")


## Held-out validation: collisions, две моды и fidelity отдельно

Один base noise повторяется для нескольких новых continuous scenes. Все methods получают
одинаковые interior noise и conditions. Parent N32 служит reference только для fidelity;
teacher collision rate не считается оптимальным. Endpoint metrics выводятся с предупреждением,
что endpoints заданы конструкцией. Test не читается.

In [ ]:
base_noise=torch.randn(SAMPLES_PER_SCENE,PATH_LENGTH-2,2,
    generator=torch.Generator().manual_seed(EVAL_SEED))
initial_noise=base_noise.repeat(EVAL_SCENES,1,1)
evaluation_conditions=validation.scene_conditions[:EVAL_SCENES].repeat_interleave(SAMPLES_PER_SCENE,dim=0)
models_to_compare={f"Teacher N{steps}":(teacher,steps) for steps in TEACHER_NFE}
models_to_compare.update({f"{name} N{STUDENT_STEPS}":(model,STUDENT_STEPS) for name,model in trained.items()})
generations,metrics={},{}
for name,(model,steps) in models_to_compare.items():
    result=sample_navigation(model,initial_noise,evaluation_conditions,steps)
    generations[name]=result
    metrics[name]=evaluate_navigation(result["samples"],evaluation_conditions,margin=COLLISION_MARGIN)
    print(name,"success:",metrics[name]["success_fraction"],"collisionfree:",metrics[name]["collision_free_fraction"],
          "both modes:",metrics[name]["mode_coverage"]["both_modes_fraction_all_conditions"])
reference=generations["Teacher N32"]["samples"][:,1:-1]
for name in metrics:
    metrics[name]["paired_interior_mse_to_teacher32"]=float((generations[name]["samples"][:,1:-1]-reference).square().mean())
assert len({r["initial_noise_sha256"] for r in generations.values()})==1
assert len({r["conditions_sha256"] for r in generations.values()})==1
atomic_json(OUTPUT_DIR/"validation_results.json",{"scope":"one_seed_exploratory_wall_validation",
    "teacher_sha256":teacher_hash,"endpoint_clamped_contract":True,"route_id_input":False,
    "noise_sha256":tensor_hash(initial_noise),"conditions_sha256":tensor_hash(evaluation_conditions),
    "evaluation_scenes":EVAL_SCENES,"samples_per_scene":SAMPLES_PER_SCENE,"models":metrics})


## Два вида анимации

**FM denoising:** как меняется весь spatial path при удалении шума. **Particle travel:**
движение A→B по уже сгенерированному финальному polyline. Это разные оси времени.
Все trajectories здесь реальные сохранённые узлы; препятствие не двигается и координаты
не нормируются заново между кадрами. В particle GIF красный — старт, фиолетовый — финиш.

In [ ]:
scene_condition=evaluation_conditions[0]
walls=scene_condition[4:].reshape(1,2,2)
start,goal=scene_condition[:2],scene_condition[2:4]
preview_names=["Teacher N1","Teacher N4","Teacher N32",*list(generations)[-2:]]
preview_histories={name:generations[name]["history"][:,:SAMPLES_PER_SCENE] for name in preview_names}
fig,axes=plt.subplots(1,len(preview_names),figsize=(4*len(preview_names),4),constrained_layout=True)
for ax,name in zip(axes,preview_names):
    plot_navigation_paths(generations[name]["samples"][:PREVIEW_PATHS],walls=walls,start=start,goal=goal,
                          ax=ax,color_by="progress",title=name+" · fixed held-out scene0")
save_figure(fig,OUTPUT_DIR/"final_paths_comparison")
plt.show()
selected=torch.cat([history[:,:PREVIEW_PATHS].flatten(0,2) for history in preview_histories.values()])
extent=max(1.1,float(selected.abs().max())+.1)
denoising_bounds=((-extent,extent),(-extent,extent))
fig,_=plot_navigation_denoising(preview_histories,walls=walls,start=start,goal=goal,
    path_indices=list(range(PREVIEW_PATHS)),progress=1.,bounds=denoising_bounds)
save_figure(fig,OUTPUT_DIR/"fm_denoising_final_frame")
plt.show()
gif_paths=[]
if SAVE_GIFS:
    flow_path=OUTPUT_DIR/"whole_path_fm_denoising.gif"
    save_navigation_denoising_gif(preview_histories,flow_path,walls=walls,start=start,goal=goal,
        path_indices=list(range(PREVIEW_PATHS)),max_frames=33,bounds=denoising_bounds)
    particle_path=OUTPUT_DIR/"particle_travel_final_path.gif"
    save_navigation_particle_gif(generations[list(generations)[-1]]["samples"][:PREVIEW_PATHS],particle_path,
        walls=walls,start=start,goal=goal,color_by="progress")
    gif_paths=[str(flow_path),str(particle_path)]
    for path in gif_paths:
        display(Image(filename=path))


In [ ]:
if SHOW_WIDGETS:
    selector=widgets.Dropdown(options=list(preview_histories),description="Method")
    frame=widgets.IntSlider(value=0,min=0,max=32,step=1,description="Euler node",continuous_update=False)
    play=widgets.Play(value=0,min=0,max=32,step=1,interval=150)
    link=widgets.jslink((play,"value"),(frame,"value"))
    canvas=widgets.Output()
    def render(change=None):
        history=preview_histories[selector.value]
        last=len(history)-1
        frame.max=play.max=last
        index=min(frame.value,last)
        with canvas:
            clear_output(wait=True)
            fig,_=plot_navigation_denoising({selector.value:history},walls=walls,start=start,goal=goal,
                path_indices=list(range(PREVIEW_PATHS)),progress=index/last,bounds=denoising_bounds)
            fig.suptitle(f"Actual Euler node{index}/{last}; denoising time, not particle-travel time")
            plt.show()
    selector.observe(render,names="value")
    frame.observe(render,names="value")
    display(widgets.VBox([widgets.HBox([selector,play,frame]),canvas]))
    render()


## Reload и варианты

Для32→2: новый `OUTPUT_DIR`, `STUDENT_STEPS=2`, повторить scene calibration и оба student loops.
Для1000updates задайте также `WARMUP=100`. `LAMBDA_W=0` проверяет paired совпадение сMSE.
Смена representation, модели или conditioner — новый эксперимент с новыми weights/calibration.
Continuous geometry8 должна сохраняться в каждом forward; route ID не добавляется как shortcut.
Validation failure честно остаётся failure: endpoint reconstruction или низкая fidelity-MSE
не заменяют collision-free success и обе допустимые моды. Этот notebook не читает final/test.

In [ ]:
restored=PointDenoiser.from_pretrained(OUTPUT_DIR/"M0/ema_model",local_files_only=True,
    low_cpu_mem_usage=False).to(device).eval().requires_grad_(False)
restored_result=sample_navigation(restored,initial_noise,evaluation_conditions,STUDENT_STEPS)
assert torch.equal(restored_result["samples"],generations[f"M0 N{STUDENT_STEPS}"]["samples"])
assert model_hash(teacher)==teacher_hash
print("Native reload matches; endpoint equality is by construction, not learned performance.")
